In [1]:
import json
import pickle
from pathlib import Path

import numpy as np

import sys
sys.path.append("workflow/scripts")
from helpers import fit_clorinn, ensure_parent, clorinn_to_dict
from fit_cv_model import load_cv_data, heldout_metrics

In [23]:
cv_input = f"/gpfs/commons/groups/knowles_lab/data/PsychGen/analysis/clorinn/cv_input/zscore_cv.npz"
method = "nnm"
nucnorm = float("256")
max_iter = 1000
svd_max_iter = 50
model_out = f"/gpfs/commons/groups/knowles_lab/data/PsychGen/analysis/clorinn/cross_validation/models/nnm_cv_model_r128.pkl"
metrics_out = f"/gpfs/commons/groups/knowles_lab/data/PsychGen/analysis/clorinn/cross_validation/metrics/nnm_cv_metrics_r128.json"

In [24]:
ztrue, ztrain, zmask = load_cv_data(cv_input)

In [25]:
model = fit_clorinn(ztrain, method, nucnorm, max_iter=max_iter, svd_max_iter=svd_max_iter, debug = True, svd_method = 'power')

2026-04-10 17:23:05,765 | clorinn.optimize.frankwolfe              | INFO    | Iteration 0. Step size 1.000. Duality Gap 294547
2026-04-10 17:23:21,294 | clorinn.optimize.frankwolfe              | INFO    | Iteration 7. Step size 0.002. Duality Gap 44.0857
2026-04-10 17:23:21,295 | clorinn.optimize.frankwolfe              | INFO    | Relative difference in objective function converged below tolerance.


In [26]:
model.st_list_

[1.0,
 1.0,
 0.3848586659130175,
 0.05524565201463994,
 0.021346761651667944,
 0.005124512586815275,
 0.021383329947677623,
 0.0017408396608179489,
 0.0017067479993651067]

In [27]:
model.dg_list_

[inf,
 294547.0113128257,
 50434.60295034081,
 2497.020932621822,
 971.4263861011391,
 225.14445349633183,
 603.352445972507,
 70.38994148845916,
 44.08574560892686]

In [28]:
ztrain_filled = np.nan_to_num(ztrain, 0)
np.linalg.norm(model.X, ord = 'fro') / np.linalg.norm(ztrain, ord = 'fro')

0.05155922813602729

In [30]:
max(None, 20)

TypeError: '>' not supported between instances of 'int' and 'NoneType'

In [9]:
# -- Save results ------
metrics = heldout_metrics(ztrue, model.X, zmask)
metrics.update(
    {
        "method": method,
        "nucnorm": nucnorm,
        "max_iter": max_iter,
        "n_iter" : len(model.steps),
    }
)
model_dict = clorinn_to_dict(model)

ensure_parent(model_out)
ensure_parent(metrics_out)

with open(model_out, "wb") as handle:
    pickle.dump(model_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(metrics_out, "w") as handle:
    json.dump(metrics, handle, indent=2, sort_keys=True)

print(f"Finished model fit at nucnorm={nucnorm:g}")
print(f"Held-out MSE: {metrics['heldout_mse']:.10f}")

Finished model fit at nucnorm=128
Held-out MSE: 2.9186867574


In [17]:
ztrain_old = ztrain.copy()